# Experimento Big Data: esquema inferido vs esquema declarado
Se leen los mismos CSV con `inferSchema` y con un esquema declarado (`StructType`) en escalas crecientes a partir
de la línea base (250 filas reales de ModaEstilo_DW): 100, 500, 10 000, 100 000, 1 000 000, 5 000 000 y
10 000 000 de filas. En cada escala se ejecuta la misma operación (5 repeticiones) y se comparan el tiempo,
los registros procesados, las filas por segundo, el tipo de dato y la consistencia del total de importes.

Requisito: haber subido `modaestilo_hechos.csv` a `/Volumes/workspace/moda/datos`.
Duración aproximada: 10 a 20 minutos (la celda 2 genera cerca de 2 GB de CSV).

In [ ]:
# 1. Configuración
import time, statistics
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, LongType, IntegerType, DateType, DecimalType)

ruta = "/Volumes/workspace/moda/datos"
carpeta_exp = f"{ruta}/exp_esquema"
VOLUMENES = [100, 500, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]
REPETICIONES = 5

# Esquema declarado: cada columna con su tipo correcto
esquema = StructType([
    StructField("tipo_operacion", StringType(), True),
    StructField("id_operacion", LongType(), True),
    StructField("id_detalle", LongType(), True),
    StructField("nro_documento", StringType(), True),
    StructField("fuente", StringType(), True),
    StructField("fecha", DateType(), True),
    StructField("id_cliente", IntegerType(), True),
    StructField("distrito", StringType(), True),
    StructField("id_trabajador", IntegerType(), True),
    StructField("cargo", StringType(), True),
    StructField("id_articulo", IntegerType(), True),
    StructField("articulo", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("canal", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio_unitario", DecimalType(12, 2), True),
    StructField("importe", DecimalType(12, 2), True),
    StructField("estado", StringType(), True),
])

base = spark.read.format("csv").option("header", "true").schema(esquema).load(f"{ruta}/modaestilo_hechos.csv")
n_base = base.count()
total_base = base.agg(F.sum("importe")).collect()[0][0]       # línea base para revisar la consistencia
print("Versión de Spark:", spark.version)
print("Filas base:", n_base, "| Total de importes base:", total_base)

In [ ]:
# 2. Generar un CSV por volumen (copias de los datos reales con identificadores distintos)
t_inicio = time.perf_counter()
for n in VOLUMENES:
    t0 = time.perf_counter()
    k = -(-n // n_base)                                         # copias necesarias
    df = (base.crossJoin(spark.range(k).withColumnRenamed("id", "copia"))
              .withColumn("id_operacion", F.col("id_operacion") + F.col("copia") * 1_000_000)
              .withColumn("id_detalle", F.col("id_detalle") + F.col("copia") * 1_000_000))
    if k * n_base > n:                                          # solo se recorta si sobran filas (100 filas)
        df = df.orderBy("copia", "id_detalle").limit(n)
    (df.drop("copia").coalesce(1).write.mode("overwrite").option("header", "true")
       .csv(f"{carpeta_exp}/filas_{n}"))
    print(f"CSV generado: {n:>11,} filas en {time.perf_counter() - t0:6.1f} s", flush=True)

print(f"Generación total: {(time.perf_counter() - t_inicio) / 60:.1f} min")
display(dbutils.fs.ls(carpeta_exp))

In [ ]:
# 3. Funciones de lectura y medición
def leer_inferido(n):
    return (spark.read.format("csv").option("header", "true").option("inferSchema", "true")
            .load(f"{carpeta_exp}/filas_{n}"))

def leer_declarado(n):
    return (spark.read.format("csv").option("header", "true").schema(esquema)
            .load(f"{carpeta_exp}/filas_{n}"))

def indicador(df):
    """Ingreso por categoría y tipo de operación (obliga a leer todo el archivo)."""
    return df.groupBy("categoria", "tipo_operacion").agg(F.sum("importe").alias("ingreso")).collect()

def medir(lector, n):
    tiempos = []
    for _ in range(REPETICIONES):
        t0 = time.perf_counter()
        indicador(lector(n))                                    # lectura + inferencia (si hay) + cálculo
        tiempos.append(time.perf_counter() - t0)
    return statistics.median(tiempos), statistics.stdev(tiempos)

def tamano_mb(n):
    """Peso en disco del CSV generado (prueba de que los datos existen)."""
    return round(sum(f.size for f in dbutils.fs.ls(f"{carpeta_exp}/filas_{n}")) / 1024 / 1024, 1)

def total_esperado(n):
    """Total que debe salir en cada escala: copias completas de la línea base más las filas sobrantes."""
    copias, resto = divmod(n, n_base)
    total = total_base * copias
    if resto:                                                   # solo con 100 filas (menos que la base)
        total += base.orderBy("id_detalle").limit(resto).agg(F.sum("importe")).collect()[0][0]
    return total

In [ ]:
# 4. Ejecución del experimento
RESULTADOS = []
t_inicio = time.perf_counter()
for n in VOLUMENES:
    t0 = time.perf_counter()
    t_inf, d_inf = medir(leer_inferido, n)
    t_dec, d_dec = medir(leer_declarado, n)

    di, dd = leer_inferido(n), leer_declarado(n)
    tipos = dict(di.dtypes)
    total_inf = di.agg(F.sum("importe")).collect()[0][0]         # suma con el tipo inferido
    total_dec = dd.agg(F.sum("importe")).collect()[0][0]         # suma con DECIMAL(12,2): exacta
    esperado = total_esperado(n)
    leidas = dd.count()

    RESULTADOS.append({
        "filas": n,
        "filas_leidas": leidas,
        "tamano_mb": tamano_mb(n),
        "inferido_s": round(t_inf, 3), "inferido_desv": round(d_inf, 3),
        "declarado_s": round(t_dec, 3), "declarado_desv": round(d_dec, 3),
        "diferencia_pct": round((t_inf - t_dec) / t_inf * 100, 1),
        "filas_s_inferido": round(n / t_inf), "filas_s_declarado": round(n / t_dec),
        "tipo_fecha_inferido": tipos["fecha"], "tipo_importe_inferido": tipos["importe"],
        "tipo_id_inferido": tipos["id_detalle"],
        "total_inferido": repr(float(total_inf)),                  # se ve el redondeo del tipo double
        "total_declarado": str(total_dec),
        "total_esperado": str(esperado),
        "consistente": "Sí" if leidas == n and total_dec == esperado else "No",
        "diferencia_total": float(total_inf) - float(total_dec),
    })
    r = RESULTADOS[-1]
    print(f"{n:>11,} filas | {r['tamano_mb']:>8} MB | inferSchema {t_inf:7.3f} s | StructType {t_dec:7.3f} s | "
          f"{r['diferencia_pct']} % | {r['filas_s_declarado']:>10,} filas/s | consistente: {r['consistente']} | "
          f"medido en {time.perf_counter() - t0:6.1f} s", flush=True)

print(f"Tiempo total del experimento: {(time.perf_counter() - t_inicio) / 60:.1f} min")

In [ ]:
# 5. Bitácora de resultados (tabla Delta)
bitacora = spark.createDataFrame(RESULTADOS).select(
    "filas", "filas_leidas", "tamano_mb", "inferido_s", "inferido_desv", "declarado_s", "declarado_desv",
    "diferencia_pct", "filas_s_inferido", "filas_s_declarado",
    "tipo_fecha_inferido", "tipo_importe_inferido", "tipo_id_inferido",
    "total_inferido", "total_declarado", "total_esperado", "consistente", "diferencia_total")
(bitacora.write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("workspace.moda.bitacora_exp_esquema"))
display(bitacora.orderBy("filas"))

In [ ]:
%sql
-- 6. Gráfico de líneas: X filas (escala logarítmica), Y inferido_s y declarado_s
SELECT filas, inferido_s, declarado_s
FROM workspace.moda.bitacora_exp_esquema
ORDER BY filas

In [ ]:
# 7. Esquemas obtenidos con el volumen más grande
leer_inferido(VOLUMENES[-1]).printSchema()
leer_declarado(VOLUMENES[-1]).printSchema()